#### Add the root directory to the system path

In [0]:
%sql
create schema goldcatalog.goldschema

In [0]:
import os
import sys
project_path = os.path.join(os.getcwd(),'..','..','..','..')
sys.path.append(project_path)

In [0]:
from utils.transformation import reusable
from pyspark.sql.functions import *
from pyspark.sql.types import *
df_object = reusable()

### DimArtist
#### Load -> Transform -> dump

In [0]:
df_artist = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
        .option('cloudFiles.schemaLocation','abfss://silver@azureprojectetestorage.dfs.core.windows.net/DimArtist/checkpoint')\
            .load('abfss://bronze@azureprojectetestorage.dfs.core.windows.net/sqldata/DimArtist')

In [0]:
df_artist = df_object.dropColumn(df_artist, ['_rescued_data'])
df_artist = df_artist.dropDuplicates(['artist_id'])

In [0]:
df_artist.writeStream.format('delta')\
    .option('checkpointLocation','abfss://silver@azureprojectetestorage.dfs.core.windows.net/DimArtist/checkpoint')\
        .trigger(once=True)\
            .option('path','abfss://silver@azureprojectetestorage.dfs.core.windows.net/DimArtist/data')\
                .toTable('silvercatalog.silverschema.DimArtist')

### DimDate
#### Load -> Transform -> dump

In [0]:
df_date = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
        .option('cloudFiles.schemaLocation','abfss://silver@azureprojectetestorage.dfs.core.windows.net/DimDate/checkpoint')\
            .load('abfss://bronze@azureprojectetestorage.dfs.core.windows.net/sqldata/DimDate')

In [0]:
df_date = df_object.dropColumn(df_date, ['_rescued_data'])
df_date = df_date.dropDuplicates(['date_key'])

In [0]:
df_date.writeStream.format('delta')\
    .option('checkpointLocation','abfss://silver@azureprojectetestorage.dfs.core.windows.net/DimDate/checkpoint')\
        .trigger(once=True)\
            .option('path','abfss://silver@azureprojectetestorage.dfs.core.windows.net/DimDate/data')\
                .toTable('silvercatalog.silverschema.DimDate')

### DimTrack
#### Load -> Transform -> dump

In [0]:
df_track = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
        .option('cloudFiles.schemaLocation','abfss://silver@azureprojectetestorage.dfs.core.windows.net/DimTrack/checkpoint')\
            .load('abfss://bronze@azureprojectetestorage.dfs.core.windows.net/sqldata/DimTrack')

In [0]:
df_track = df_track.withColumn('duration_flag', when(col('duration_sec') < 150, 'short')\
                               .when(col('duration_sec') < 300 , 'medium')\
                                   .otherwise('long'))
df_track = df_track.withColumn('track_name', regexp_replace(col('track_name'),'-',' '))
df_track = df_object.dropColumn(df_track, ['_rescued_data'])
df_track = df_track.dropDuplicates(['track_id'])

In [0]:
df_track.writeStream.format('delta')\
    .option('checkpointLocation','abfss://silver@azureprojectetestorage.dfs.core.windows.net/DimTrack/checkpoint')\
        .trigger(once=True)\
            .option('path','abfss://silver@azureprojectetestorage.dfs.core.windows.net/DimTrack/data')\
                .toTable('silvercatalog.silverschema.DimTrack')

### DimUser
#### Load -> Transform -> dump

In [0]:
df_user = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
        .option('cloudFiles.schemaLocation','abfss://silver@azureprojectetestorage.dfs.core.windows.net/DimUser/checkpoint')\
            .load('abfss://bronze@azureprojectetestorage.dfs.core.windows.net/sqldata/DimUser')

In [0]:
df_user = df_object.dropColumn(df_user, ['_rescued_data'])
# df_user = df_user.dropDuplicates(['user_id'])
df_user = df_user.withColumn('user_name', upper('user_name'))

In [0]:
df_user.writeStream.format('delta')\
    .option('checkpointLocation','abfss://silver@azureprojectetestorage.dfs.core.windows.net/DimUser/checkpoint')\
        .trigger(once=True)\
            .option('path','abfss://silver@azureprojectetestorage.dfs.core.windows.net/DimUser/data')\
                .toTable('silvercatalog.silverschema.DimUser')

### FactStream
#### Load -> Transform -> dump

In [0]:
df_stream = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
        .option('cloudFiles.schemaLocation','abfss://silver@azureprojectetestorage.dfs.core.windows.net/FactStream/checkpoint')\
            .load('abfss://bronze@azureprojectetestorage.dfs.core.windows.net/sqldata/FactStream')

In [0]:
df_stream = df_object.dropColumn(df_stream, ['_rescued_data'])
df_stream = df_stream.dropDuplicates(['stream_id'])

In [0]:
df_stream.writeStream.format('delta')\
    .option('checkpointLocation','abfss://silver@azureprojectetestorage.dfs.core.windows.net/FactStream/checkpoint')\
        .trigger(once=True)\
            .option('path','abfss://silver@azureprojectetestorage.dfs.core.windows.net/FactStream/data')\
                .toTable('silvercatalog.silverschema.FactStream')